# 02. Training the Anomaly Detection Model
## Fan Predictive Maintenance System

This notebook demonstrates training an **Isolation Forest** on healthy operational baseline data to detect anomalous behaviors.

In [ ]:
import sys
import os
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.anomaly_detection import AnomalyDetector

print("Modules imported successfully.")

### 1. Data Preprocessing & Healthy Baseline Extraction

In [ ]:
preprocessor = DataPreprocessor(config_path='../config/config.yaml')
preprocessor.load_data('../data/predictive_maintenance_dataset.csv')
preprocessor.validate_data()

baseline = preprocessor.extract_baseline()
print(f"Extracted {len(baseline)} healthy baseline samples from {len(preprocessor.data)} total samples.")

### 2. Feature Engineering

In [ ]:
engineer = FeatureEngineer(config_path='../config/config.yaml')
baseline_eng, feature_names = engineer.engineer_features(baseline)
X_baseline = baseline_eng[feature_names].values

data_eng, _ = engineer.engineer_features(preprocessor.data)
X_all = data_eng[feature_names].values
y_all = preprocessor.get_labels()

print(f"Feature matrix shape: {X_all.shape}")
print(f"Engineered Features: {feature_names}")

### 3. Model Training (Isolation Forest)

In [ ]:
detector = AnomalyDetector(config_path='../config/config.yaml')
detector.train(X_baseline)

# Evaluate on full dataset
metrics = detector.evaluate(X_all, y_all)
print("\n=== Model Evaluation Results ===")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}" if isinstance(v, float) else f"{k.upper()}: {v}")

### 4. ROC-AUC & Confusion Matrix Visualizations

In [ ]:
scores = detector.predict_proba(X_all)
fpr, tpr, thresholds = roc_curve(y_all, scores)
auc_val = roc_auc_score(y_all, scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#2980b9', lw=2, label=f'Isolation Forest (AUC = {auc_val:.4f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[0].set_title('Receiver Operating Characteristic (ROC)')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# Confusion Matrix
preds_binary = (detector.predict(X_all) == -1).astype(int)
cm = confusion_matrix(y_all, preds_binary)
ConfusionMatrixDisplay(cm, display_labels=['Healthy', 'Anomaly']).plot(ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix')

plt.tight_layout()
plt.show()

### 5. Anomaly Score Distribution

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(scores[y_all == 0], bins=30, alpha=0.7, color='#2ecc71', label='Healthy Samples (y=0)', density=True)
plt.hist(scores[y_all == 1], bins=30, alpha=0.7, color='#e74c3c', label='Anomaly Samples (y=1)', density=True)
plt.axvline(x=0.6, color='black', linestyle='--', label='Alert Threshold (0.6)')
plt.title('Calibrated Anomaly Score Distributions')
plt.xlabel('Anomaly Probability Score')
plt.ylabel('Density')
plt.legend()
plt.show()

### 6. Save Model Artifact

In [ ]:
os.makedirs('../models', exist_ok=True)
detector.save('../models/isolation_forest_model.pkl')
print("Trained model saved to ../models/isolation_forest_model.pkl")